# Lesson 01: Exercise Solutions

Worked answers to the five exercises at the end of
[`01_linear_regression.ipynb`](01_linear_regression.ipynb). Try them yourself first —
the failures are the instructive part.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from linear_regression import (
    compute_cost, compute_gradient, gradient_descent, predict,
    normal_equation, zscore_normalize,
)

rng = np.random.default_rng(0)
plt.rcParams["figure.figsize"] = (6, 4)
plt.rcParams["axes.grid"] = True

# the same dataset as the lesson: y = 3.5x - 2 + noise
m = 100
X = rng.uniform(0, 10, size=(m, 1))
y = 3.5 * X[:, 0] - 2.0 + rng.normal(0, 1.5, size=m)

---
# Exercise 1: Break it on purpose

> Remove the `.copy()` in `gradient_descent`, or update `b` using the already-updated
> `w`. What changes, and does the numerical gradient check still pass?

## 1a. Removing `.copy()`

The subtlety: `w = w - alpha * dj_dw` **rebinds** `w` to a brand-new array, so the
caller's array is never touched — dropping `.copy()` changes nothing. The bug only bites
if you also write the update *in place*:

```python
w = w - alpha * dj_dw     # rebinds -> caller safe even without .copy()
w -= alpha * dj_dw        # mutates in place -> caller's array is destroyed
```

`-=` is the natural thing to write, and it is faster (no temporary allocation). That is
what makes this trap easy to fall into.

In [ ]:
def gd_inplace(X, y, w, b, alpha, num_iters):
    # NO .copy(), and an in-place update
    for _ in range(num_iters):
        dj_dw, dj_db = compute_gradient(X, y, w, b)
        w -= alpha * dj_dw          # <-- mutates the caller's array
        b -= alpha * dj_db
    return w, b


w0 = np.zeros(1)
print("before first fit :  w0 =", w0)

w_1, b_1 = gd_inplace(X, y, w0, 0.0, alpha=0.01, num_iters=10)
print("after  first fit :  w0 =", w0.round(4), " <-- our starting point was overwritten")

# Run the "same" experiment again with the "same" initial guess:
w_2, b_2 = gd_inplace(X, y, w0, 0.0, alpha=0.01, num_iters=10)
print(f"\nrun 1: w = {w_1[0]:.4f}, b = {b_1:+.4f}")
print(f"run 2: w = {w_2[0]:.4f}, b = {b_2:+.4f}   <-- identical call, different answer")

# The correct version, run under exactly the same protocol:
w0_ok = np.zeros(1)
c1_w, c1_b, _ = gradient_descent(X, y, w0_ok, 0.0, 0.01, 10)
c2_w, c2_b, _ = gradient_descent(X, y, w0_ok, 0.0, 0.01, 10)
print(f"\ncorrect: run 1 b = {c1_b:+.4f}, run 2 b = {c2_b:+.4f}   (reproducible)")
print("correct: w0 afterwards =", w0_ok, " (untouched)")

Read the numbers carefully. `w0` went from `[0.]` to a half-trained value, so the second
run silently started from wherever the first one finished. Here `w` happens to converge
within a few iterations either way, so the damage shows up in `b`, which *does* reset to
`0.0` each call, because a Python float is passed by value while a NumPy array is passed
by reference. Half your state resets and half of it doesn't. Nothing errors,
nothing warns — you just get an experiment that is not the one you thought you ran. This
class of bug is brutal in a loop over hyperparameters, where every configuration after
the first inherits the previous one's parameters and the comparison becomes meaningless.

Watch it wreck a learning-rate sweep:

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

for ax, share in zip(axes, [True, False]):
    w_shared = np.zeros(1)                      # one array reused across the sweep
    for alpha in [0.002, 0.005, 0.02]:
        w_start = w_shared if share else np.zeros(1)
        costs = []
        w_run, b_run = w_start, 0.0
        for _ in range(60):
            dj_dw, dj_db = compute_gradient(X, y, w_run, b_run)
            w_run -= alpha * dj_dw              # in-place
            b_run -= alpha * dj_db
            costs.append(compute_cost(X, y, w_run, b_run))
        ax.plot(costs, label=f"$\\alpha$ = {alpha}")
    ax.set_yscale("log"); ax.set_xlabel("iteration"); ax.set_ylabel("$J$"); ax.legend()
    ax.set_title("shared array (buggy)" if share else "fresh array each time (correct)")

plt.tight_layout(); plt.show()

On the left, $\alpha = 0.005$ and $\alpha = 0.02$ look wonderful because they began
almost converged. You would conclude the wrong thing about every learning rate you tested.

**The fix** is the defensive `.copy()` that is already in `linear_regression.py`. A
function that receives an array should not modify it unless that is explicitly its job.

## 1b. Updating `b` with the already-updated `w`

Now the other bug: compute `w`'s gradient, update `w`, then compute `b`'s gradient at
the **new** `w`.

In [ ]:
def gd_sequential(X, y, w, b, alpha, num_iters):
    # non-simultaneous: b's gradient is measured at the already-updated w
    w, b, hist = np.asarray(w, float).copy(), float(b), []
    for _ in range(num_iters):
        dj_dw, _ = compute_gradient(X, y, w, b)
        w = w - alpha * dj_dw                      # w moves first
        _, dj_db = compute_gradient(X, y, w, b)    # then b's gradient at the NEW w
        b = b - alpha * dj_db
        hist.append((w[0], b, compute_cost(X, y, w, b)))
    return w, b, hist


# short runs, for plotting the paths below
w_ok, b_ok, h_ok = gradient_descent(X, y, np.zeros(1), 0.0, 0.01, 400, record_every=1)
w_sq, b_sq, h_sq = gd_sequential(X, y, np.zeros(1), 0.0, 0.01, 400)
print("after 400 iterations (neither has converged yet):")
print(f"  simultaneous : w = {w_ok[0]:.6f}, b = {b_ok:.6f}")
print(f"  sequential   : w = {w_sq[0]:.6f}, b = {b_sq:.6f}")

# run both all the way out
w_ok_f, b_ok_f, _ = gradient_descent(X, y, np.zeros(1), 0.0, 0.01, 20_000)
w_sq_f, b_sq_f, _ = gd_sequential(X, y, np.zeros(1), 0.0, 0.01, 20_000)
w_star, b_star = normal_equation(X, y)
print("\nafter 20,000 iterations:")
print(f"  simultaneous (correct) : w = {w_ok_f[0]:.6f}, b = {b_ok_f:.6f}")
print(f"  sequential   (buggy)   : w = {w_sq_f[0]:.6f}, b = {b_sq_f:.6f}")
print(f"  normal equation (exact): w = {w_star[0]:.6f}, b = {b_star:.6f}")

**Both converge to the same place.** That is the uncomfortable lesson: on a convex
problem the buggy version still works, because at the optimum both partial derivatives
are zero, so both updates stop at the same point. The bug is invisible in the final answer.

It is only visible in the *path*:

In [ ]:
w_grid = np.linspace(-0.5, 5.5, 120)
b_grid = np.linspace(-3.5, 1.0, 120)
WW, BB = np.meshgrid(w_grid, b_grid)
JJ = np.array([[compute_cost(X, y, np.array([w]), b) for w in w_grid] for b in b_grid])

path_sq = np.array([[h[0], h[1]] for h in h_sq])

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].contour(WW, BB, JJ, levels=np.logspace(-0.3, 3.0, 20), cmap="viridis", alpha=0.6)
axes[0].plot(path_sq[:, 0], path_sq[:, 1], "r.-", ms=2, lw=1, label="sequential (buggy)")
axes[0].plot(3.5, -2.0, "k*", ms=13)
axes[0].set_xlabel("$w$"); axes[0].set_ylabel("$b$"); axes[0].legend(); axes[0].set_title("Descent path")

axes[1].plot(h_ok["cost"], label="simultaneous (correct)")
axes[1].plot([h[2] for h in h_sq], "--", label="sequential (buggy)")
axes[1].set_yscale("log"); axes[1].set_xlabel("iteration"); axes[1].set_ylabel("$J$")
axes[1].legend(); axes[1].set_title("Learning curves")
plt.tight_layout(); plt.show()

The sequential version takes a *different route* it is doing roughly one and a half
steps per iteration, since `b` gets the benefit of `w`'s fresh move. (This has a name:
it is Gauss–Seidel rather than Jacobi iteration.) Here that is harmless and even slightly
faster per iteration. On a non-convex problem, or with momentum, or across a mini-batch,
"harmless" stops being guaranteed — and you will have no test that catches it.

## Does the numerical gradient check still pass?

**Yes — and that is the whole point of the exercise.**

In [ ]:
def numerical_gradient(X, y, w, b, eps=1e-6):
    dj_dw = np.array([
        (compute_cost(X, y, w + eps * np.eye(len(w))[j], b)
         - compute_cost(X, y, w - eps * np.eye(len(w))[j], b)) / (2 * eps)
        for j in range(len(w))
    ])
    dj_db = (compute_cost(X, y, w, b + eps) - compute_cost(X, y, w, b - eps)) / (2 * eps)
    return dj_dw, dj_db


w_t, b_t = np.array([1.3]), 0.4
print("analytic :", compute_gradient(X, y, w_t, b_t))
print("numeric  :", numerical_gradient(X, y, w_t, b_t))
print("\nIdentical -> the gradient check passes with both bugs present.")

Neither bug lives in `compute_gradient`, so a gradient check cannot see them. Gradient
checking validates **the derivative**, not **the optimiser**. The bugs above are caught
by different tools:

| bug | what catches it |
|---|---|
| in-place mutation of the caller's array | a test asserting the input is unchanged after `fit` |
| non-simultaneous update | comparing against the closed-form solution, or a step-by-step trace |
| wrong derivative | the numerical gradient check |

Know what each of your tests can and cannot see.

---
# Exercise 2: Mean absolute error

> Swap the squared error for $\frac{1}{m}\sum|f(x^{(i)}) - y^{(i)}|$. Derive its
> gradient (careful at zero), implement it, and compare how the two costs react to an outlier.

## Derivation

$$J_{\text{MAE}}(w,b) = \frac{1}{m}\sum_{i=1}^{m}\left|f_{w,b}(x^{(i)}) - y^{(i)}\right|$$

Since $\frac{d}{du}|u| = \operatorname{sign}(u)$ for $u \neq 0$, the chain rule gives

$$\frac{\partial J}{\partial w_j} = \frac{1}{m}\sum_{i=1}^{m}\operatorname{sign}\!\left(f(x^{(i)}) - y^{(i)}\right)x_j^{(i)}
\qquad
\frac{\partial J}{\partial b} = \frac{1}{m}\sum_{i=1}^{m}\operatorname{sign}\!\left(f(x^{(i)}) - y^{(i)}\right)$$

**Careful at zero.** $|u|$ has a kink at $u = 0$, no derivative exists there. We use a
*subgradient*: any value in $[-1, 1]$ is valid, and we pick $0$, which is what
`np.sign(0)` returns anyway. In practice a residual is never exactly zero on real data,
and if it is, that example simply contributes nothing to this step.

Contrast the two gradients and the difference in behaviour falls out immediately:

$$\underbrace{(f - y)\,x_j}_{\text{MSE: scales with the error}}
\qquad\text{vs.}\qquad
\underbrace{\operatorname{sign}(f - y)\,x_j}_{\text{MAE: only the direction}}$$

MSE lets a badly-wrong point shout proportionally louder. MAE gives every point exactly
one vote. That single difference is the whole story of the outlier experiment below.

In [ ]:
def compute_cost_mae(X, y, w, b):
    return float(np.mean(np.abs(predict(X, w, b) - y)))


def compute_gradient_mae(X, y, w, b):
    m = X.shape[0]
    s = np.sign(predict(X, w, b) - y)      # +1 / -1 / 0, NOT the error itself
    return (X.T @ s) / m, float(s.sum() / m)


# gradient check (residuals are never exactly 0 on random data, so the kink is not hit)
w_t, b_t, eps = np.array([1.3]), 0.4, 1e-6
ana_w, ana_b = compute_gradient_mae(X, y, w_t, b_t)
num_w = (compute_cost_mae(X, y, w_t + eps, b_t) - compute_cost_mae(X, y, w_t - eps, b_t)) / (2 * eps)
num_b = (compute_cost_mae(X, y, w_t, b_t + eps) - compute_cost_mae(X, y, w_t, b_t - eps)) / (2 * eps)
print(f"dJ/dw  analytic {ana_w[0]:10.6f}   numeric {num_w:10.6f}")
print(f"dJ/db  analytic {ana_b:10.6f}   numeric {num_b:10.6f}")

### MAE needs a decaying learning rate

The MAE gradient does **not** shrink as you approach the optimum, it is made of $\pm 1$
terms, so it stays roughly the same size forever. With a fixed $\alpha$ the iterates
bounce around the minimum in a permanent step-sized orbit instead of settling.

The standard fix is to decay the step size, e.g. $\alpha_t = \alpha_0 / (1 + t/\tau)$.

In [ ]:
def gradient_descent_general(X, y, w, b, alpha, num_iters, grad_fn, cost_fn, decay=None):
    w, b, hist = np.asarray(w, float).copy(), float(b), []
    for t in range(num_iters):
        a = alpha if decay is None else alpha / (1 + t / decay)
        dj_dw, dj_db = grad_fn(X, y, w, b)
        w = w - a * dj_dw
        b = b - a * dj_db
        hist.append(cost_fn(X, y, w, b))
    return w, b, hist


_, _, h_fixed = gradient_descent_general(X, y, np.zeros(1), 0.0, 0.05, 600,
                                         compute_gradient_mae, compute_cost_mae)
_, _, h_decay = gradient_descent_general(X, y, np.zeros(1), 0.0, 0.05, 600,
                                         compute_gradient_mae, compute_cost_mae, decay=50)

plt.plot(h_fixed, label="fixed $\\alpha$ = 0.05")
plt.plot(h_decay, label="decaying $\\alpha_t = 0.05/(1+t/50)$")
plt.xlabel("iteration"); plt.ylabel("$J_{MAE}$"); plt.legend()
plt.title("MAE: a fixed step size never settles"); plt.ylim(1.1, 1.6); plt.show()

### The outlier experiment

One house is mis-recorded, a data-entry error puts its price 60 units too high. Nothing
else changes.

In [ ]:
X_out = np.vstack([X, [[9.5]]])
y_out = np.append(y, 3.5 * 9.5 - 2.0 + 60.0)      # one wildly wrong label, near the edge

fits = {}
for name, Xd, yd in [("clean", X, y), ("with outlier", X_out, y_out)]:
    w_mse, b_mse, _ = gradient_descent(Xd, yd, np.zeros(1), 0.0, 0.01, 6000)
    w_mae, b_mae, _ = gradient_descent_general(Xd, yd, np.zeros(1), 0.0, 0.4, 6000,
                                               compute_gradient_mae, compute_cost_mae, decay=200)
    fits[name] = {"MSE": (w_mse[0], b_mse), "MAE": (w_mae[0], b_mae)}

print(f"{'':<16}{'MSE w':>10}{'MSE b':>10}{'MAE w':>10}{'MAE b':>10}")
for name, f in fits.items():
    print(f"{name:<16}{f['MSE'][0]:>10.3f}{f['MSE'][1]:>10.3f}{f['MAE'][0]:>10.3f}{f['MAE'][1]:>10.3f}")
print(f"{'-> shift':<16}"
      f"{fits['with outlier']['MSE'][0]-fits['clean']['MSE'][0]:>10.3f}"
      f"{fits['with outlier']['MSE'][1]-fits['clean']['MSE'][1]:>10.3f}"
      f"{fits['with outlier']['MAE'][0]-fits['clean']['MAE'][0]:>10.3f}"
      f"{fits['with outlier']['MAE'][1]-fits['clean']['MAE'][1]:>10.3f}")

# sanity check: our MAE fit really does beat the MSE fit *at minimising MAE*
w_m, b_m = fits["clean"]["MSE"]
w_a, b_a = fits["clean"]["MAE"]
print(f"\non clean data, J_MAE at the MSE fit = {compute_cost_mae(X, y, np.array([w_m]), b_m):.5f}")
print(f"                 J_MAE at the MAE fit = {compute_cost_mae(X, y, np.array([w_a]), b_a):.5f}  <- lower, as it must be")

xs = np.linspace(0, 10, 2).reshape(-1, 1)
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for ax, (name, Xd, yd) in zip(axes, [("clean", X, y), ("with outlier", X_out, y_out)]):
    ax.scatter(Xd, yd, s=16, alpha=0.5)
    for loss, style in [("MSE", "r-"), ("MAE", "b--")]:
        w_f, b_f = fits[name][loss]
        ax.plot(xs, xs[:, 0] * w_f + b_f, style, label=f"{loss}: w={w_f:.2f}")
    ax.set_title(name); ax.set_xlabel("$x$"); ax.legend()
axes[0].set_ylabel("$y$")
plt.tight_layout(); plt.show()

**The result.** Read the `-> shift` row: it is how far each fit moved when the single bad
point was added. The MSE slope shifts by $0.26$; the MAE slope shifts by $0.004$ — nearly
two orders of magnitude less. The MSE line visibly tilts to chase the outlier while the
MAE line barely notices it.

Why: that outlier's residual is about $60$. Squaring makes its contribution to the cost
roughly $3600$  larger than all hundred good points put together, so the optimiser buys
a big reduction by bending toward it. Under MAE it contributes $60$, and its *gradient*
contribution is $\operatorname{sign} = 1$: exactly one vote, the same as every other point.

Two details worth not glossing over:

- **Placement matters.** I put the outlier at $x = 9.5$, the edge of the data. A point far
  from the centre has **leverage**  it torques the slope. The same bad point at $x = 5$
  (the middle) would mostly shift the intercept and leave the slope nearly intact. Try it.
- **The two clean fits are not identical**, and shouldn't be. They minimise different
  quantities: MSE estimates the conditional *mean*, MAE the conditional *median*. On a
  finite noisy sample those differ. The printed check underneath confirms our MAE fit is
  genuinely converged — it achieves a lower $J_{\text{MAE}}$ than the MSE fit does.

The trade-off is not free: MAE is not differentiable everywhere, has no closed-form
solution, and converges more slowly. This is a genuine modelling decision — *do I believe
my extreme values are signal or noise?* MSE says signal, MAE says noise. (Huber loss is
the common compromise: squared near zero, linear in the tails.)

---
# Exercise 3 : A convergence test

> Stop when $J$ improves by less than $10^{-9}$ between iterations instead of running a
> fixed `num_iters`. How many iterations does each $\alpha$ actually need?

In [ ]:
def gradient_descent_until_converged(X, y, w, b, alpha, tol=1e-9, max_iters=2_000_000):
    w, b = np.asarray(w, float).copy(), float(b)
    prev = compute_cost(X, y, w, b)
    for i in range(1, max_iters + 1):
        dj_dw, dj_db = compute_gradient(X, y, w, b)
        w = w - alpha * dj_dw
        b = b - alpha * dj_db
        cost = compute_cost(X, y, w, b)
        if not np.isfinite(cost) or cost > prev:
            return w, b, i, "DIVERGING"    # cost growing: not convergence!
        if prev - cost < tol:              # improvement too small to be worth continuing
            return w, b, i, "converged"
        prev = cost
    return w, b, max_iters, "hit max_iters"


w_star, b_star = normal_equation(X, y)
print(f"{'alpha':>8}{'iters':>10}{'status':>14}{'w':>10}{'b':>10}{'|error|':>12}")
for alpha in [0.0005, 0.005, 0.02, 0.0295, 0.06]:
    w_c, b_c, n_iter, status = gradient_descent_until_converged(X, y, np.zeros(1), 0.0, alpha)
    err = max(abs(w_c[0] - w_star[0]), abs(b_c - b_star))
    print(f"{alpha:>8}{n_iter:>10}{status:>14}{w_c[0]:>10.4f}{b_c:>10.4f}{err:>12.2e}")

Three things worth noticing:

1. **The iteration counts span orders of magnitude.** A well-chosen $\alpha$ is worth
   more than any amount of waiting.
2. **Stopping early is not the same as being correct.** The `|error|` column is the
   distance from the exact solution. A *tiny improvement per step* does not mean *close
   to the optimum*, it can equally mean tiny steps. With a small $\alpha$ the test fires
   while the parameters are still measurably off. This is the standard trap with
   relative-improvement stopping criteria.
3. **A diverging run must be detected separately.** `prev - cost` is negative when the
   cost is *growing*, which is less than `tol` — so a naive test declares "converged" on
   a run that is exploding. Hence the explicit `np.isfinite` guard and the sign check.

A more honest criterion tests the thing you actually care about — the gradient, which is
zero at the optimum by definition:

In [ ]:
def gradient_descent_grad_norm(X, y, w, b, alpha, tol=1e-6, max_iters=2_000_000):
    w, b = np.asarray(w, float).copy(), float(b)
    for i in range(1, max_iters + 1):
        dj_dw, dj_db = compute_gradient(X, y, w, b)
        if np.sqrt(dj_dw @ dj_dw + dj_db ** 2) < tol:
            return w, b, i
        w = w - alpha * dj_dw
        b = b - alpha * dj_db
    return w, b, max_iters


print(f"{'alpha':>8}{'iters':>10}{'|error|':>12}")
for alpha in [0.0005, 0.005, 0.02, 0.0295]:
    w_c, b_c, n_iter = gradient_descent_grad_norm(X, y, np.zeros(1), 0.0, alpha)
    print(f"{alpha:>8}{n_iter:>10}{max(abs(w_c[0]-w_star[0]), abs(b_c-b_star)):>12.2e}")

Now every $\alpha$ lands on the same answer to the same accuracy, and the only difference
is how long it took, which is exactly what we wanted to measure.

---
# Exercise 4 : Polynomial features

> Fit $y = 0.5x^2 + 3$ by feeding the model a second column $x^2$. The model is still
> linear in its parameters.

The key idea: "linear regression" means **linear in $w$ and $b$**, not linear in $x$. We
are free to invent new features from the old ones. Fitting

$$f(x) = w_1 x + w_2 x^2 + b$$

is still linear regression, it is just linear regression on the design matrix
$X = [\,x \;\; x^2\,]$. Every line of code we already wrote applies unchanged.

In [ ]:
m3 = 120
x_raw = rng.uniform(-6, 6, m3)
y_poly = 0.5 * x_raw ** 2 + 3.0 + rng.normal(0, 1.5, m3)

X_poly = np.column_stack([x_raw, x_raw ** 2])       # <-- the entire trick
print("column ranges:  x in [%.1f, %.1f]   x^2 in [%.1f, %.1f]"
      % (x_raw.min(), x_raw.max(), (x_raw**2).min(), (x_raw**2).max()))

Note those ranges: $x^2$ spans a far wider range than $x$. This is precisely the badly
scaled situation from section 7 of the lesson, and it is *inherent* to polynomial
features, $x^3$, $x^4$ make it dramatically worse. **Always scale polynomial features.**

In [ ]:
X_poly_n, mu_p, sigma_p = zscore_normalize(X_poly)
w_p, b_p, hist_p = gradient_descent(X_poly_n, y_poly, np.zeros(2), 0.0, 0.05, 4000)

# undo the scaling to read the coefficients in original units
w_orig = w_p / sigma_p
b_orig = b_p - np.sum(w_p * mu_p / sigma_p)
w_pe, b_pe = normal_equation(X_poly, y_poly)        # closed form on the raw features
print(f"gradient descent:  y = {w_orig[0]:+.3f} x {w_orig[1]:+.3f} x^2 {b_orig:+.3f}")
print(f"normal equation :  y = {w_pe[0]:+.3f} x {w_pe[1]:+.3f} x^2 {b_pe:+.3f}")
print( "true            :  y = +0.000 x +0.500 x^2 +3.000")
print("\nGradient descent matches the closed form; both differ slightly from the true"
      "\ncoefficients because we only have 120 noisy samples -- that gap is sampling"
      "\nerror, not an optimisation failure.")

grid = np.linspace(-6.5, 6.5, 200)
G = np.column_stack([grid, grid ** 2])
Gn, _, _ = zscore_normalize(G, mu_p, sigma_p)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].scatter(x_raw, y_poly, s=16, alpha=0.5, label="data")
axes[0].plot(grid, predict(Gn, w_p, b_p), "r-", lw=2, label="quadratic fit")
w_lin, b_lin, _ = gradient_descent(x_raw.reshape(-1, 1), y_poly, np.zeros(1), 0.0, 0.01, 4000)
axes[0].plot(grid, grid * w_lin[0] + b_lin, "g--", label="straight line (underfits)")
axes[0].set_xlabel("$x$"); axes[0].set_ylabel("$y$"); axes[0].legend(); axes[0].set_title("Same algorithm, curved fit")

axes[1].plot(hist_p["cost"]); axes[1].set_yscale("log")
axes[1].set_xlabel("iteration"); axes[1].set_ylabel("$J$"); axes[1].set_title("Learning curve")
plt.tight_layout(); plt.show()

The learned $x$ coefficient comes out near zero, correctly discovering that the data has
no linear term. The straight-line fit is nearly flat and hopeless — a vivid picture of
**underfitting**: the model is not wrong so much as incapable of expressing the truth.

This is the doorway to a lot of machine learning. Bases you can swap in the same way:
$\sqrt{x}$, $\log x$, $\sin x$, products of features $x_1x_2$ (interactions), or
one-hot indicators for categories. Choosing them is *feature engineering*; the fitting
machinery never changes. The natural next question — "why not add $x^{15}$ and fit
anything?" — is overfitting, which is lesson 03.

---
# Exercise 5 : Stochastic gradient descent

> Update using one random example at a time instead of all $m$. Plot the learning curve —
> why is it noisy, and does it still get there?

Batch gradient descent computes the exact gradient using all $m$ examples, then takes
one step. **Stochastic** gradient descent estimates it from a single random example:

$$w := w - \alpha\left(f_{w,b}(x^{(i)}) - y^{(i)}\right)x^{(i)} \qquad i \text{ drawn at random}$$

That single-example gradient is an **unbiased estimator** of the true gradient — right on
average, but wrong on any given step. Cheap and noisy versus expensive and exact.

In [ ]:
def sgd(X, y, w, b, alpha, num_epochs, batch_size=1, decay=None, seed=1):
    rng_local = np.random.default_rng(seed)
    w, b = np.asarray(w, float).copy(), float(b)
    m = X.shape[0]
    hist, evals = [], []
    n_seen = 0

    for epoch in range(num_epochs):
        order = rng_local.permutation(m)               # reshuffle each epoch
        for start in range(0, m, batch_size):
            idx = order[start:start + batch_size]
            a = alpha if decay is None else alpha / (1 + epoch / decay)
            dj_dw, dj_db = compute_gradient(X[idx], y[idx], w, b)   # same function!
            w = w - a * dj_dw
            b = b - a * dj_db
            n_seen += len(idx)
            hist.append(compute_cost(X, y, w, b))       # full cost, for plotting only
            evals.append(n_seen)
    return w, b, np.array(hist), np.array(evals)


w_sgd, b_sgd, h_sgd, e_sgd = sgd(X, y, np.zeros(1), 0.0, alpha=0.01, num_epochs=30)
w_mb,  b_mb,  h_mb,  e_mb  = sgd(X, y, np.zeros(1), 0.0, alpha=0.01, num_epochs=30, batch_size=10)
w_dec, b_dec, h_dec, e_dec = sgd(X, y, np.zeros(1), 0.0, alpha=0.03, num_epochs=30, decay=3)

w_bat, b_bat, h_bat = gradient_descent(X, y, np.zeros(1), 0.0, 0.01, 30, record_every=1)

print(f"batch (30 iters, 3000 example-gradients) : w = {w_bat[0]:.4f}, b = {b_bat:.4f}")
print(f"SGD   (30 epochs, 3000 example-gradients): w = {w_sgd[0]:.4f}, b = {b_sgd:.4f}")
print(f"mini-batch 10                            : w = {w_mb[0]:.4f}, b = {b_mb:.4f}")
print(f"SGD + decaying alpha                     : w = {w_dec[0]:.4f}, b = {b_dec:.4f}")
print(f"exact                                    : w = {w_star[0]:.4f}, b = {b_star:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(e_sgd, h_sgd, lw=0.7, alpha=0.8, label="SGD (batch size 1)")
axes[0].plot(e_mb, h_mb, lw=1.0, label="mini-batch (10)")
axes[0].plot(np.arange(1, 31) * m, h_bat["cost"], "k-", lw=2, label="batch (all 100)")
axes[0].axhline(compute_cost(X, y, w_star, b_star), color="r", ls=":", label="optimum")
axes[0].set_xscale("log"); axes[0].set_yscale("log")
axes[0].set_xlabel("example-gradients computed (equal compute)"); axes[0].set_ylabel("$J$")
axes[0].legend(fontsize=8); axes[0].set_title("Per unit of work, SGD gets there first")

axes[1].plot(h_sgd[-600:], lw=0.7, label="fixed $\\alpha$")
axes[1].plot(h_dec[-600:], lw=0.9, label="decaying $\\alpha$")
axes[1].axhline(compute_cost(X, y, w_star, b_star), color="r", ls=":", label="optimum")
axes[1].set_xlabel("update (last 600)"); axes[1].set_ylabel("$J$")
axes[1].legend(fontsize=8); axes[1].set_title("Late training: the noise ball")

plt.tight_layout(); plt.show()

## Why is it noisy, and does it still get there?

**Why noisy.** Each step follows the gradient of *one example*, not the average. A point
above the line and a point below it pull in opposite directions, so consecutive steps
partly undo one another. The cost bounces because the estimator has high variance —
correct on average, wrong every time.

**Does it get there?** Look at the left panel, where the x-axis is *work done* rather
than iterations. At equal compute SGD reaches a good solution far sooner: batch
descent spends 100 example-gradients to take one step, SGD takes 100 steps with the same
budget. Early on, a noisy step in roughly the right direction beats a perfect step you
had to pay 100× for.

But look at the right panel. With a **fixed** $\alpha$ it never truly converges — it
settles into a *noise ball* around the optimum whose radius is set by $\alpha$. It keeps
jittering forever. **Decaying** $\alpha$ shrinks that ball as training proceeds and lets
the iterates actually settle. The classical Robbins–Monro conditions for convergence are

$$\sum_t \alpha_t = \infty \qquad \sum_t \alpha_t^2 < \infty$$

— steps big enough in total to reach anywhere, but shrinking fast enough to settle.
$\alpha_t = \alpha_0/(1+t/\tau)$ satisfies both.

**Mini-batching** is the practical middle ground and what essentially all real training
uses: batch sizes of 32–512 average away most of the variance while still taking many
steps per pass over the data, and the batch is a matrix multiply that GPUs devour.

Notice that `sgd` above calls the very same `compute_gradient` passing `X[idx], y[idx]`
instead of `X, y`. Batch, mini-batch, and stochastic gradient descent differ *only* in how
many rows you hand it.

---
## Recap

| exercise | the transferable lesson |
|---|---|
| 1 | Gradient checks validate the gradient, not the optimiser. Never mutate a caller's array. |
| 2 | The loss encodes what you believe about your data. Squared error assumes outliers are signal. |
| 3 | "Stopped improving" ≠ "reached the optimum". Test the gradient norm, and detect divergence explicitly. |
| 4 | Linear regression is linear *in the parameters*. Engineer features freely — and scale them. |
| 5 | A noisy cheap gradient beats an exact expensive one early; decay the step size to settle. |

On to lesson 02, logistic regression.